# Site Interaction Analysis: launch-month corrected

This notebook fixes the month-indexing bug in the original analysis.

The key rule is now simple:
- launch month = the month that contains `operational_start_date`
- the month before launch = 0
- earlier months are negative
- all pre/post windows are anchored to the **launch month**, not the exact launch day and not the panel's global January 2024 calendar

That means if a site starts on `2024-04-12`, then `2024-04` is month 1 for that site.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

DATA_DIR = Path('/Users/dhruvsood/sonnysDataCollection/hypothesis-testing')
OUT_DIR = DATA_DIR / 'interaction_outputs'
OUT_DIR.mkdir(exist_ok=True)

sys.path.append(str(DATA_DIR))

from site_interaction_analysis_lib import (
    METRICS,
    build_pair_deltas,
    build_pair_event_profile,
    build_panel,
    build_quad_deltas,
    build_sites,
    build_triple_deltas,
    build_triple_event_profile,
    configure_plotting,
    curate_outputs,
    find_pairs,
    find_quads,
    find_triples,
    fmt_pct,
    interaction_keep_set,
    plot_any_new_operator_effect,
    plot_existing_single_new_multi_trend,
    plot_market_saturation,
    plot_new_multi_three_body_trend,
    plot_pair_examples_all,
    plot_quad_examples_all,
    plot_triple_examples_all,
    prepare_interaction_dirs,
    write_interaction_readme,
    write_report,
)

configure_plotting()
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

MAX_NEIGHBOR_MILES = 10.0
PRE_POST_WINDOW = 6
OUT = prepare_interaction_dirs(OUT_DIR)
write_interaction_readme(OUT_DIR)
print('Writing artefacts to', OUT_DIR)
print('  plots/two_body|three_body|four_body|aggregate')
print('  data/', OUT['data'].relative_to(OUT_DIR))
print('  report/', OUT['report'].relative_to(OUT_DIR))


## 1. Build the monthly panel and correct the site-relative month number

The original `less_than-2yrs.csv` month numbering was tied to the panel calendar, which made some sites start at month 1 **before** they were operational.

Here we recompute site age from scratch using `operational_start_date`.


In [ ]:
panel, validation = build_panel(DATA_DIR)

print(f"Unified panel: {validation['sites']} sites, {validation['rows']:,} site-month rows")
print(f"Panel coverage: {validation['month_min'].date()} to {validation['month_max'].date()}")
print(f"  lt2 (less_than-2yrs.csv): {validation['lt2_sites']} sites, {validation['lt2_rows']:,} rows")
print(f"  gt2 (more_than-2yrs_monthly.csv): {validation['gt2_sites']} sites, {validation['gt2_rows']:,} rows")
print(f"lt2 rows whose month number changed after the fix: {validation['lt2_rows_with_changed_month_number']:,}")
print(f"lt2 sites affected: {validation['lt2_sites_with_changed_month_number']}")
print(f"Raw lt2 prelaunch rows still present in source data: {validation['lt2_prelaunch_rows']:,}")

display(Markdown(
    f"**Why this matters**\n\n"
    f"- We found **{validation['lt2_rows_with_changed_month_number']:,}** rows where the old `month_number` did not match the launch-month logic.\n"
    f"- Those rows span **{validation['lt2_sites_with_changed_month_number']}** different newer sites.\n"
    f"- From now on, the notebook uses `site_month_number`, not the raw `month_number` from the CSV."
))

validation['examples']


## 2. Build the site table and identify nearby launch pairs

We still define the local market using straight-line distance, with a same-ZIP preference when a same-ZIP option exists inside the radius.


In [ ]:
sites, site_lookup, site_distances = build_sites(panel)
pairs_df = find_pairs(sites, site_distances, max_neighbor_miles=MAX_NEIGHBOR_MILES, pre_buffer_months=PRE_POST_WINDOW)

print(f"Sites with coordinates: {len(sites)}")
print(f"Sites with a real launch month: {int(sites['has_launch_month'].sum())}")
print(f"Two-body candidate pairs inside {MAX_NEIGHBOR_MILES:.0f} miles: {len(pairs_df)}")
print(f"Same-ZIP candidate pairs selected: {int(pairs_df['zip_match'].sum())}")

pairs_df.head(10)


## 3. Two-body results

The event month is the new site's **launch month**, and the existing site is compared across the 6 months before vs the 6 months after that launch month.


In [ ]:
pair_deltas = build_pair_deltas(panel, pairs_df, pre_post_window=PRE_POST_WINDOW, min_months=3)
pair_profile = build_pair_event_profile(pair_deltas, panel, window=PRE_POST_WINDOW)

pair_deltas.to_csv(OUT['data'] / 'two_body_pair_deltas.csv', index=False)
plot_pair_examples_all(pair_deltas, panel, OUT['two_body'] / 'examples_all_sites.png')
plot_existing_single_new_multi_trend(
    pair_deltas, panel, OUT['two_body'] / 'avg_existing_single_new_multi_trend.png', PRE_POST_WINDOW
)

print(f"Usable two-body pairs: {len(pair_deltas)}")
display(Markdown(
    f"**Headline read**\n\n"
    f"- Median existing-site total change: **{fmt_pct(pair_deltas['existing_pct_wash_count_total'].median())}**\n"
    f"- Median combined-market total change: **{fmt_pct(pair_deltas['combined_pct_wash_count_total'].median())}**\n"
    f"- Saved under `plots/two_body/` and `data/`"
))
display(Image(filename=str(OUT['two_body'] / 'examples_all_sites.png')))


## 4. Three-body results

Now C is the newest launch month, and A/B are the two closest older sites within 10 miles that were already live at least 6 months earlier.


In [ ]:
triples_df = find_triples(sites, site_distances, max_neighbor_miles=MAX_NEIGHBOR_MILES, pre_buffer_months=PRE_POST_WINDOW)
triple_deltas = build_triple_deltas(panel, triples_df, pre_post_window=PRE_POST_WINDOW, min_months=3)
triple_profile = build_triple_event_profile(triple_deltas, panel, window=PRE_POST_WINDOW)

triple_deltas.to_csv(OUT['data'] / 'three_body_triple_deltas.csv', index=False)
plot_triple_examples_all(triple_deltas, panel, OUT['three_body'] / 'examples_all_sites.png')
plot_new_multi_three_body_trend(
    triple_deltas, panel, OUT['three_body'] / 'avg_new_multi_intro_trend.png', PRE_POST_WINDOW
)

print(f"Usable three-body triples: {len(triple_deltas)}")
display(Markdown(
    f"**Headline read**\n\n"
    f"- Median A-site total change after C launches: **{fmt_pct(triple_deltas['A_pct_wash_count_total'].median())}**\n"
    f"- Median B-site total change after C launches: **{fmt_pct(triple_deltas['B_pct_wash_count_total'].median())}**\n"
    f"- Median full A+B+C market total change: **{fmt_pct(triple_deltas['region_pct_wash_count_total'].median())}**\n"
    f"- Saved under `plots/three_body/` and `data/`"
))
display(Image(filename=str(OUT['three_body'] / 'examples_all_sites.png')))


## 4b. Four-body results

D is the newest launch month. A/B/C are the three closest older sites within 10 miles that were already live at least 6 months earlier, with at least 6 months between each launch.

In [ ]:
quads_df = find_quads(sites, site_distances, max_neighbor_miles=MAX_NEIGHBOR_MILES, pre_buffer_months=PRE_POST_WINDOW)
quad_deltas = build_quad_deltas(panel, quads_df, pre_post_window=PRE_POST_WINDOW, min_months=3)

quad_deltas.to_csv(OUT['data'] / 'four_body_quad_deltas.csv', index=False)
plot_quad_examples_all(quad_deltas, panel, OUT['four_body'] / 'examples_all_sites.png')

print(f"Candidate four-body quads: {len(quads_df)} | usable: {len(quad_deltas)}")
display(Markdown(
    f"**Headline read**\n\n"
    f"- Median A-site total change after D launches: **{fmt_pct(quad_deltas['A_pct_wash_count_total'].median())}**\n"
    f"- Median B-site total change after D launches: **{fmt_pct(quad_deltas['B_pct_wash_count_total'].median())}**\n"
    f"- Median C-site total change after D launches: **{fmt_pct(quad_deltas['C_pct_wash_count_total'].median())}**\n"
    f"- Median full A+B+C+D market total change: **{fmt_pct(quad_deltas['region_pct_wash_count_total'].median())}**\n"
    f"- Saved under `plots/four_body/` and `data/`"
))
display(Image(filename=str(OUT['four_body'] / 'examples_all_sites.png')))

## 5. Report and prune outputs

Writes `report/site_interaction_report.md` and deletes anything outside `plots/`, `data/`, and `README.md`.


In [ ]:
plot_any_new_operator_effect(
    pair_deltas, panel, OUT['aggregate'] / 'any_new_operator_effect.png', PRE_POST_WINDOW
)
plot_market_saturation(pair_deltas, OUT['aggregate'] / 'market_saturation_threshold.png')

report_path = write_report(
    OUT_DIR,
    panel,
    validation,
    pair_deltas,
    triple_deltas,
    pair_profile,
    triple_profile,
    max_neighbor_miles=MAX_NEIGHBOR_MILES,
    pre_post_window=PRE_POST_WINDOW,
)
write_interaction_readme(OUT_DIR)
curate_outputs(OUT_DIR)

print('Kept artefacts:', len(interaction_keep_set()))
for rel in sorted(interaction_keep_set()):
    print(' ', rel)
print('Report:', report_path.relative_to(OUT_DIR))
